# 1. LangChain高级用法
1. bind_tools():为模型绑定工具
2. with_structured_output():按照指定的数据结构（Schema）解析模型输出，方便程序调用

## 1.1 bind_tools():
bind_tools() 不负责真正执行 Python 函数。它只是向模型提供工具定义，并允许模型在响应中返回 tool_calls。

LangChain支持多种工具定义方式，包括：
- 使用 @tool 装饰器定义的 LangChain 工具。
- 普通 Python 函数（由集成转换其定义）。
- Pydantic 模型类。
- 符合要求的 OpenAI Tool Calling 格式字典。

In [7]:
# 使用字典定义工具,OpenAI Function Calling风格的字典描述工具
'''
OpenAI Function Calling风格的字典描述工具，使用字典定义工具，包含工具的类型、名称、描述和参数信息。以下是一个示例代码，展示了如何定义一个天气查询工具：

'''
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model(
    model = os.getenv("LLM_MODEL"),
    api_key = os.getenv("OPENAI_API_KEY"),
    base_url = os.getenv("OPEN_BASE_URL"),
    model_provider="openai"
)

# 描述一个天气查询工具
tools = [
    {
        'type': 'function',
        'description':"获取指定城市的天气信息",
        'function':{
            'name': 'get_weather',
            'parameters':{
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，例如杭州、北京"
                    }
                },
                'required': ['city']
            }
        }
    }
]
# 将工具绑定到模型
model_with_tools = model.bind_tools(tools)
# 请求模型使用工具
response = model_with_tools.invoke(
    "请使用get_weather工具查询杭州的天气"
)
# 查看工具调用
if response.tool_calls:
    for tool_call in response.tool_calls:
        print(f'工具名称：{tool_call["name"]}')
        print(f'工具类型：{tool_call["type"]}')
        print(f'工具参数：{tool_call["args"]}')
        print(f'调用id：{tool_call["id"]}')
else:
    print("没有工具调用")

工具名称：get_weather
工具类型：tool_call
工具参数：{'city': '杭州'}
调用id：call_373ed5b57e0f40f7ab4b3ed6


In [8]:
# 使用@tool定义工具

from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    '''
    查询指定城市的模拟天气。
    Args:
        city: 城市名称，如杭州、北京。
    '''
    weather_data = {
        "杭州": "晴，25°C",
        "北京": "多云，20°C",
        "上海": "小雨，22°C",
        "广州": "雷阵雨，28°C"
    }
    return weather_data.get(
        city,
        f"抱歉，无法查询 {city} 的天气信息。"
    )

# 绑定工具
model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke(
    "请使用get_weather工具查询北京的天气"
)
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_f43fb66d5e3f4a9083e1cac8', 'type': 'tool_call'}]


In [5]:
# 使用Pydantic模型描述工具参数
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

load_dotenv()

class GetWeather(BaseModel):
    '''查询指定城市的模拟天气'''
    city: str = Field(description="城市名称，如杭州、北京")
    unit: str = Field(
        default = "C",
        description="温度单位，默认为摄氏度(C)，可选值为'C'或'F'"
    )

class Calculate(BaseModel):
    '''计算简单的数学表达式'''
    expression: str = Field(description="数学表达式，例如 2 + 2")

model = init_chat_model(
    model = os.getenv("LLM_MODEL"),
    api_key = os.getenv("OPENAI_API_KEY"),
    base_url = os.getenv("OPEN_BASE_URL"),
    model_provider="openai"
)
model_with_tools = model.bind_tools([GetWeather, Calculate])

response = model_with_tools.invoke(
    "查询北京的模拟天气，并计算123乘以456。"
)

for tool_call in response.tool_calls:
    print("工具：", tool_call["name"])
    print("参数：", tool_call["args"])

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


工具： GetWeather
参数： {'city': '北京'}
工具： Calculate
参数： {'expression': '123 * 456'}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## 1.2 with_structured_output()


In [6]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

load_dotenv()


# 定义结构化输出格式
class PersonInfo(BaseModel):
    """从文本中提取人物信息。"""

    name: str = Field(
        description="人物姓名"
    )

    age: int = Field(
        description="年龄"
    )

    occupation: str = Field(
        description="职业"
    )

    skills: list[str] = Field(
        description="技能列表"
    )


# 初始化模型
model = init_chat_model(
    model = os.getenv("LLM_MODEL"),
    api_key = os.getenv("OPENAI_API_KEY"),
    base_url = os.getenv("OPEN_BASE_URL"),
    model_provider="openai"
)

# 使用 Pydantic Schema
structured_model = model.with_structured_output(
    PersonInfo
)

text = (
    "张三今年28岁，是一名全栈工程师，"
    "精通 Python、React 和 Docker。"
)

result = structured_model.invoke(
    "请根据文本提取人物信息，不要编造：\n" + text
)

print("姓名：", result.name)
print("年龄：", result.age)
print("职业：", result.occupation)
print("技能：", result.skills)

print("返回类型：", type(result))

姓名： 张三
年龄： 28
职业： 全栈工程师
技能： ['Python', 'React', 'Docker']
返回类型： <class '__main__.PersonInfo'>
